# 03 — Phase 3: Target Engineering


## 1 — Bootstrap


In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
SRC = ROOT / 'src'
assert SRC.exists(), f'Could not find src/ at {SRC}'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

import config
from dataset import build_dataset
from stationary import add_stationary_features, get_stationary_features
from evaluation import walk_forward_evaluate
from targets import (build_directional_dataset, compare_targets,
                     forward_return, target_autocorrelation,
                     threshold_target, volatility_scaled_target)
from abstention import (abstention_report, brier_score, calibration_table,
                        calibrate_probabilities, coverage_accuracy_curve,
                        evaluate_with_abstention)

pd.set_option('display.width', 140)
plt.rcParams['figure.dpi'] = 110

def rf(n=500, leaf=20):
    return RandomForestClassifier(n_estimators=n, min_samples_leaf=leaf,
                                  random_state=config.RANDOM_STATE, n_jobs=-1)

BASE = {}
for key in config.list_stocks():
    d = build_dataset(key, with_sentiment=False, save=False, verbose=False)
    BASE[key] = add_stationary_features(d).dropna().reset_index(drop=True)
    print(f'{key:10s} {len(BASE[key]):5d} rows')

FEATURES = get_stationary_features(BASE['TCS'])
print(f'\nStationary features: {len(FEATURES)}')


## 2 — The size of the moves being labelled


In [ ]:
d = BASE['TCS']
fwd = forward_return(d, 1).dropna()

bands = {
    'under 0.25%': (fwd.abs() < 0.0025).mean(),
    '0.25% - 0.5%': ((fwd.abs() >= 0.0025) & (fwd.abs() < 0.005)).mean(),
    '0.5% - 1.0%': ((fwd.abs() >= 0.005) & (fwd.abs() < 0.010)).mean(),
    '1.0% - 2.0%': ((fwd.abs() >= 0.010) & (fwd.abs() < 0.020)).mean(),
    'over 2.0%': (fwd.abs() >= 0.020).mean(),
}

band_df = pd.DataFrame({'share_of_days': pd.Series(bands).round(4)})
band_df['cumulative'] = band_df['share_of_days'].cumsum().round(4)
print(band_df.to_string())
print()
print(f'Median absolute daily move : {fwd.abs().median():.4%}')
print(f'Days moving under 0.5%     : {(fwd.abs() < 0.005).mean():.1%}')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(fwd * 100, bins=120, color='#5C6BC0', alpha=0.85)
ax.axvspan(-0.5, 0.5, color='#F44336', alpha=0.18,
           label='under 0.5% — near-unlabellable')
ax.axvline(0, color='#424242', lw=1)
ax.set_xlim(-6, 6)
ax.set_xlabel('Next-day return (%)')
ax.set_ylabel('Number of days')
ax.set_title('TCS — distribution of next-day returns', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'return_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 3 — Comparing target definitions


In [ ]:
specs = {
    'binary 1d (Phase 1/2)': lambda x: threshold_target(x, 1, 0.0),
    'fixed 0.5% 1d': lambda x: threshold_target(x, 1, 0.005),
    'fixed 1.0% 1d': lambda x: threshold_target(x, 1, 0.010),
    'vol-scaled k=0.5 1d': lambda x: volatility_scaled_target(x, 1, 0.5),
    'vol-scaled k=1.0 1d': lambda x: volatility_scaled_target(x, 1, 1.0),
    'vol-scaled k=0.5 5d': lambda x: volatility_scaled_target(x, 5, 0.5),
}

for key in BASE:
    print(f'\n{key}')
    print(compare_targets(BASE[key], specs).to_string())


## 4 — Fixed vs volatility-scaled thresholds across stocks


In [ ]:
rows = []
for key, d in BASE.items():
    fwd_k = forward_return(d, 1).dropna()
    rows.append({
        'stock': key,
        'median_abs_move': round(float(fwd_k.abs().median()), 5),
        'coverage_fixed_0.5%': round(float((fwd_k.abs() > 0.005).mean()), 4),
        'coverage_vol_k0.5': round(float(
            (volatility_scaled_target(d, 1, 0.5).dropna() != 0).mean()), 4),
    })

print(pd.DataFrame(rows).set_index('stock').to_string())


## 5 — The overlapping-target trap


In [ ]:
rows = []
for h in [1, 2, 3, 5, 10, 20]:
    overlap = build_directional_dataset(BASE['TCS'], horizon=h, k=0.5,
                                        non_overlapping=False)
    nonover = build_directional_dataset(BASE['TCS'], horizon=h, k=0.5,
                                        non_overlapping=True)
    rows.append({
        'horizon': h,
        'n_overlapping': len(overlap),
        'autocorr_overlapping': round(overlap['Target'].autocorr(1), 4),
        'n_nonoverlapping': len(nonover),
        'autocorr_nonoverlapping': round(nonover['Target'].autocorr(1), 4),
    })

overlap_df = pd.DataFrame(rows).set_index('horizon')
print(overlap_df.to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(overlap_df.index, overlap_df['autocorr_overlapping'],
        'o-', color='#F44336', lw=2, label='Overlapping targets')
ax.plot(overlap_df.index, overlap_df['autocorr_nonoverlapping'],
        'o-', color='#26A69A', lw=2, label='Non-overlapping targets')
ax.axhline(0, color='#424242', lw=1, ls=':')
ax.set_xlabel('Horizon (trading days)')
ax.set_ylabel('Target autocorrelation (lag 1)')
ax.set_title('Label independence vs horizon', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'target_autocorrelation.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 6 — Persistence baseline inflation


In [ ]:
rows = []
for h in [1, 3, 5, 10]:
    for nonov in [False, True]:
        d = build_directional_dataset(BASE['TCS'], horizon=h, k=0.5,
                                      non_overlapping=nonov)
        if len(d) < 200:
            continue
        r = walk_forward_evaluate(d, FEATURES, rf(), n_splits=5,
                                  embargo=max(52 // h, 5))
        b = r.baselines.mean()
        rows.append({
            'horizon': h,
            'non_overlapping': nonov,
            'n': len(d),
            'model': round(r.per_fold['accuracy'].mean(), 4),
            'majority': round(b['majority'], 4),
            'persistence': round(b['persistence'], 4),
        })

persist_df = pd.DataFrame(rows)
print(persist_df.to_string(index=False))


## 7 — Horizon and threshold sweep


In [ ]:
sweep = []
for h in [1, 3, 5, 10]:
    for k in [0.0, 0.5, 1.0, 1.5]:
        d = build_directional_dataset(BASE['TCS'], horizon=h, k=k,
                                      non_overlapping=True)
        if len(d) < 200:
            continue
        try:
            r = walk_forward_evaluate(d, FEATURES, rf(), n_splits=5,
                                      embargo=max(52 // h, 5))
        except ValueError:
            continue
        s = r.summary()
        base = r.baselines.mean().max()
        sweep.append({
            'horizon': h,
            'k': k,
            'n': len(d),
            'accuracy': round(s['accuracy_mean'], 4),
            'std': round(s['accuracy_std'], 4),
            'best_baseline': round(base, 4),
            'edge': round(s['accuracy_mean'] - base, 4),
        })

sweep_df = pd.DataFrame(sweep)
print(sweep_df.to_string(index=False))
print()
print('Configurations whose edge exceeds their own fold std:')
print(sweep_df[sweep_df['edge'] > sweep_df['std']].to_string(index=False))


## 8 — Confidence-gated prediction


In [ ]:
CONFIG = dict(horizon=1, k=0.5, non_overlapping=True)

OOF = {}
for key, d in BASE.items():
    ds = build_directional_dataset(d, **CONFIG)
    emb = 52 if key == 'TCS' else 20
    r = walk_forward_evaluate(ds, FEATURES, rf(), n_splits=5, embargo=emb)
    OOF[key] = r
    print(f'{key}: {len(ds)} big-move rows of {len(d)} '
          f"({len(ds) / len(d):.0%} coverage), "
          f"overall acc {r.per_fold['accuracy'].mean():.4f}, "
          f'baseline {r.baselines.mean().max():.4f}')


In [ ]:
CURVES = {}
for key, r in OOF.items():
    curve = coverage_accuracy_curve(r.oof_predictions, min_samples=15)
    CURVES[key] = curve
    print(f'\n{key}')
    print(curve.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

for ax, (key, curve) in zip(axes, CURVES.items()):
    ax.plot(curve['coverage'], curve['accuracy'], 'o-',
            color='#26A69A', lw=2, label='Model accuracy')
    ax.plot(curve['coverage'], curve['subset_majority'], 'o--',
            color='#F44336', lw=1.5, label='Majority on same subset')
    ax.set_xlabel('Coverage (fraction of days acted on)')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'{key} — accuracy vs coverage', fontweight='bold')
    ax.invert_xaxis()
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'coverage_accuracy.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 9 — Probability calibration


In [ ]:
for key, r in OOF.items():
    print(f'\n{key}')
    print(f'  Brier score (raw): {brier_score(r.oof_predictions)}'
          '   [0.25 = uninformative]')
    cal = calibrate_probabilities(r.oof_predictions)
    print(f'  Brier score (isotonic): '
          f"{brier_score(cal, proba_col='y_proba_calibrated')}")
    print(calibration_table(r.oof_predictions).to_string(index=False))


## 10 — Operating point


In [ ]:
points = []
for key, r in OOF.items():
    for thresh in [0.0, 0.03, 0.05, 0.08, 0.10, 0.12]:
        res = evaluate_with_abstention(r.oof_predictions,
                                       confidence_threshold=thresh)
        res['stock'] = key
        points.append(res)

points_df = pd.DataFrame(points)[
    ['stock', 'confidence_threshold', 'coverage', 'n_committed',
     'accuracy', 'subset_majority', 'edge']]
print(points_df.to_string(index=False))


## 11 — Save Phase 3 results


In [ ]:
sweep_df.to_csv(config.REPORTS_DIR / 'phase3_sweep.csv', index=False)
overlap_df.to_csv(config.REPORTS_DIR / 'phase3_overlap.csv')
points_df.to_csv(config.REPORTS_DIR / 'phase3_operating_points.csv',
                 index=False)
for key, curve in CURVES.items():
    curve.to_csv(config.REPORTS_DIR / f'phase3_coverage_{key}.csv',
                 index=False)

summary = []
for key, r in OOF.items():
    curve = CURVES[key]
    best = curve.loc[curve['edge'].idxmax()]
    summary.append({
        'stock': key,
        'n_big_moves': int(curve.iloc[0]['n_predictions']),
        'full_coverage_accuracy': curve.iloc[0]['accuracy'],
        'full_coverage_edge': curve.iloc[0]['edge'],
        'best_coverage': best['coverage'],
        'best_accuracy': best['accuracy'],
        'best_edge': best['edge'],
        'brier': brier_score(r.oof_predictions),
    })

phase3 = pd.DataFrame(summary).set_index('stock')
phase3.to_csv(config.REPORTS_DIR / 'phase3_results.csv')
print('Saved to reports/')
print()
print(phase3.to_string())
